### Wczytywanie danych


In [ ]:
import pandas as pd

dane = pd.read_csv("advertising.csv")
X = dane[["TV","Radio","Newspaper"]]
y = dane[["Sales"]]
dane

In [ ]:
import plotly.express as px

fig1 = px.scatter(dane,x="Sales",y="TV",title= "TV ads vs Sales")
fig2 = px.scatter(dane,x="Sales",y="Radio",title= "Radio ads vs Sales")
fig3 = px.scatter(dane,x="Sales",y="Newspaper",title= "Newspaper ads vs Sales")

fig1.show()
fig2.show()
fig3.show()

In [ ]:
correlations = dane.corr()
correlations

Widzimy, że jak na wykresach i w tabeli korelacji że reklamy w TV są silnie powiązane z ilością sprzedaży, te w radio są lekko powiązane, a te w gazetach nie mają wielkiego wpływu, natomiast z tabeli korelacji wynika, również że reklamy w radio są lekko powiązane z tymi w gazetach, ale wydaje mi się że to jest tylko przypadek

In [ ]:
dane_melted = dane.melt(value_vars=['TV', 'Radio', 'Newspaper', 'Sales'], 
                    var_name='Var', value_name='Wartość')

fig = px.box(dane_melted, y='Wartość', facet_col='Var', color='Var', 
             title='')
fig.update_yaxes(matches=None)

fig.show()

Dzięki tym wykresom widzimy, że mamy dwa outliery dla wartości reklam w gazetach, które mogą zakłócić wyniki naszego modelu regresji liniowej dlatego je odrzucamy, dzięki metodzie IQR

In [ ]:
def preprocess_data(X_train,y_train):
    Q1 = X_train["Newspaper"].quantile(0.25)
    Q3 = X_train["Newspaper"].quantile(0.75)
    IQR = Q3-Q1
    lower_bound = Q1 - 1.5*IQR
    upper_bound = Q3 + 1.5*IQR
    
    maska = (X_train["Newspaper"]>=lower_bound) & (X_train["Newspaper"] <= upper_bound)
    
    
    X_train = X_train[maska]
    y_train=y_train[maska]
    
    return X_train,y_train
    
    
    
    

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.dummy import DummyRegressor

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train_clean,y_train_clean = preprocess_data(X_train,y_train)


scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test)

lr_clean = LinearRegression()
lr_scaled = LinearRegression()

lr_clean.fit(X_train_clean,y_train_clean)
lr_scaled.fit(X_train_scaled,y_train_clean)


y_pred_clean = lr_clean.predict(X_test)

y_pred_scaled = lr_scaled.predict(X_test_scaled)

mse_lr_clean = mean_squared_error(y_test,y_pred_clean)
mse_lr_scaled = mean_squared_error(y_test,y_pred_scaled)

table_data = {
    "Model" : ["Regresja Liniowa skalowana","Regresja Liniowa czysta"],
    "Błąd MSE" : [mse_lr_scaled,mse_lr_clean]
}
table = pd.DataFrame(table_data)
print(table)







Widzimy, że skalowanie nie zmienia precyzji, a tylko zmienia wagi. Obie metody są tak samo dokładne, ale wybrałbym tą skalowaną.